In [1]:
!rm -rf InkubaLM-Challenge
!git clone https://github.com/melissafasol/InkubaLM-Challenge.git
%cd InkubaLM-Challenge

Cloning into 'InkubaLM-Challenge'...
remote: Enumerating objects: 343, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 343 (delta 88), reused 51 (delta 23), pack-reused 202 (from 1)
Receiving objects: 100% (343/343), 1.18 MiB | 15.12 MiB/s, done.
Resolving deltas: 100% (215/215), done.
/content/InkubaLM-Challenge


In [2]:
!git checkout refactor-src-structure
!git pull

Branch 'refactor-src-structure' set up to track remote branch 'refactor-src-structure' from 'origin'.
Switched to a new branch 'refactor-src-structure'
Already up to date.


In [3]:
!pip install datasets
!pip install transformers datasets peft trl accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 15.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which

In [4]:
# Cell 1: Install dependencies
!pip install -q transformers accelerate peft datasets bitsandbytes trl

# Cell 2: Imports and setup
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from datasets import load_dataset, Dataset
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, Gemma3ForConditionalGeneration

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
Using device: cuda


In [5]:
import sys
sys.path.append("..")  # Add parent directory to the path

import os
from typing import List
from pathlib import Path
import numpy as np

# DO NOT EDIT
# create submission file
import pandas as pd
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
)

from utils import (
    model_function,
    eval
    )

from src import (
    data_utils,
    model_utils,
    inference,
    prompts,
    evaluation,
    data_augment
    )

import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset, concatenate_datasets, Dataset, Value, DatasetDict

from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM
from peft import PeftModel, PeftConfig
from sklearn.model_selection import train_test_split

In [6]:
#os.environ["TOKENIZERS_PARALLELISM"] = "false"

from huggingface_hub import login

try:
    from google.colab import userdata

    # Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
    # vars as appropriate for your system.
    # userdata.get("HF_TOKEN") indicates that the name of the token in the Colab env is HF_TOKEN
    os.environ["hf_token_2"] = userdata.get("hf_token_2")
except:
    os.environ["hf_token_2"] = "----"

login(token=os.environ["hf_token_2"])

token = os.environ["hf_token_2"]
if token == "----":
    print("⚠️ Warning: No Hugging Face token found. Some models may not load.")
else:
    login(token=token)

In [7]:
hf_token_2 = '..' # paste your token here
os.environ["HF_TOKEN"] = hf_token_2


In [8]:
from huggingface_hub import login
login(token=hf_token_2)  # Force login using this token


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [9]:
import os
import json

file_path = '/content/'

# Infer language from ID or langs field
def infer_lang(example):
    if example["task"] == "translation":
        return example.get("langs", "eng-swa")
    elif "swa" in example["ID"]:
        return "swahili"
    elif "hau" in example["ID"]:
        return "hausa"
    else:
        return "unknown"

# Build prompt without <lang> tokens
def format_prompt(ex):
    instruction = ex.get("instruction", "").strip()
    input_text = ex.get("input", "").strip()
    return f"{instruction}\n{input_text}".strip() if input_text else instruction

# Load and transform data
input_files = [
    os.path.join(file_path, "gemma_sentiment_soft_labels.jsonl"),
    os.path.join(file_path, "gemma_xnli_soft_labels_mc.jsonl"),
    os.path.join(file_path, "gemma_mt_distilled.jsonl"),
]

all_tasks = []
for fname in input_files:
    with open(fname) as f:
        for line in f:
            ex = json.loads(line)
            ex["lang"] = infer_lang(ex)
            ex["prompt"] = format_prompt(ex)
            all_tasks.append(ex)

# Save to final multitask JSONL
output_path = os.path.join(file_path, "multitask_distill.jsonl")
with open(output_path, "w") as f:
    for row in all_tasks:
        f.write(json.dumps(row) + "\n")

print(f"✅ Saved multitask distillation dataset to: {output_path}")


✅ Saved multitask distillation dataset to: /content/multitask_distill.jsonl


In [10]:
from datasets import Dataset

dataset = Dataset.from_list(all_tasks)


In [48]:
sentiment_xnli_raw = all_tasks[0:600]
sentiment_xnli_raw[0]

{'ID': 'ID_f3c74c7b_sentiment_test__hausa',
 'task': 'sentiment',
 'instruction': "Your task is to do sentiment analysis. Your output must be one word: positive, negative, or neutral. Don't output any explanation.",
 'input': '@user ynxu fha da kanada kudi shikenan duk kayan nan zasu iya zama naka no🧢',
 'output': 'negative',
 'soft_label': {'positive': 0.0, 'neutral': 6e-05, 'negative': 0.99844},
 'lang': 'hausa',
 'prompt': "Your task is to do sentiment analysis. Your output must be one word: positive, negative, or neutral. Don't output any explanation.\n@user ynxu fha da kanada kudi shikenan duk kayan nan zasu iya zama naka no🧢"}

In [64]:
import os
import torch
import torch.nn.functional as F
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model
from datasets import Dataset as HFDataset
from torch.utils.data import Dataset as TorchDataset

# ---------------------------
# 1. Label map
# ---------------------------
LABEL_MAP = {
    "sentiment": ["positive", "negative", "neutral"],
    "xnli": ["true", "false", "neither"]
}

# ---------------------------
# 2. Convert soft_label dicts to vectors
# ---------------------------
def preprocess_soft_labels(examples):
    processed = []
    for i, ex in enumerate(examples):
        task = ex.get("task")
        label_order = LABEL_MAP.get(task, [])
        if not label_order:
            continue

        soft_dict = ex.get("soft_label", {})
        label_vector = [float(soft_dict.get(label, 0.0) or 0.0) for label in label_order]

        if sum(label_vector) == 0:
            continue

        processed.append({
            "prompt": ex["prompt"].strip(),
            "soft_label": label_vector,
            "task": task  # include task for dynamic loss handling
        })
    return processed

# ---------------------------
# 3. Load Lelapa InkubaLM-0.4B with LoRA + 4-bit
# ---------------------------
def load_lora_model(model_name="lelapa/InkubaLM-0.4B"):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto"
    )

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model, tokenizer

# ---------------------------
# 4. Custom Trainer with KL loss over class logits
# ---------------------------
class SoftDistillationTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        soft_labels = inputs.pop("soft_label")
        task_labels = inputs.pop("task")
        outputs = model(**inputs)
        logits = outputs.logits[:, -1, :]  # shape: [batch, vocab]

        task = task_labels[0]  # assumes same task in batch
        label_tokens = LABEL_MAP[task]
        label_token_ids = tokenizer.convert_tokens_to_ids(label_tokens)
        selected_logits = logits[:, label_token_ids]  # shape: [batch, num_labels]

        student_log_probs = F.log_softmax(selected_logits, dim=-1)
        loss = F.kl_div(student_log_probs, soft_labels.to(student_log_probs.device), reduction="batchmean")
        return (loss, outputs) if return_outputs else loss

# ---------------------------
# 5. Data collator
# ---------------------------
class SoftLabelDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        return {
            "input_ids": torch.stack([f["input_ids"] for f in features]),
            "attention_mask": torch.stack([f["attention_mask"] for f in features]),
            "soft_label": torch.stack([f["soft_label"] for f in features]),
            "task": [f["task"] for f in features],  # task names (as list of str)
        }

# ---------------------------
# 6. PyTorch dataset wrapper
# ---------------------------
class TorchSoftLabelDataset(TorchDataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __getitem__(self, idx):
        item = self.dataset[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "soft_label": torch.tensor(item["soft_label"], dtype=torch.float32),
            "task": item["task"]
        }

    def __len__(self):
        return len(self.dataset)

# ---------------------------
# 7. Training pipeline
# ---------------------------
def train_soft_distilled_lora_model(raw_data, model_name="lelapa/InkubaLM-0.4B"):
    print("✅ Preprocessing data...")
    processed = preprocess_soft_labels(raw_data)
    dataset = HFDataset.from_list(processed)

    print("✅ Loading model and tokenizer...")
    global tokenizer
    model, tokenizer = load_lora_model(model_name)

    def tokenize_function(example):
        encoding = tokenizer(
            example["prompt"],
            padding="max_length",
            truncation=True,
            max_length=512
        )
        return {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
            "soft_label": example["soft_label"],
            "task": example["task"]
        }

    print("✅ Tokenizing dataset...")
    tokenized = dataset.map(tokenize_function, remove_columns=dataset.column_names)

    print("✅ Filtering out incomplete samples...")
    tokenized = tokenized.filter(
        lambda x: x.get("soft_label") is not None and all(v is not None for v in x["soft_label"])
    )
    print(f"✅ Final dataset size: {len(tokenized)} examples")

    print("✅ Wrapping in PyTorch dataset...")
    torch_dataset = TorchSoftLabelDataset(tokenized)

    training_args = TrainingArguments(
        output_dir="./soft_distill_inkuba_output",
        per_device_train_batch_size=4,
        num_train_epochs=3,
        gradient_accumulation_steps=2,
        logging_steps=10,
        save_steps=50,
        logging_dir="./logs",
        report_to="none",
        evaluation_strategy="no",
        do_eval=False,
        remove_unused_columns=False,
    )

    print("🚀 Starting training...")
    trainer = SoftDistillationTrainer(
        model=model,
        args=training_args,
        train_dataset=torch_dataset,
        eval_dataset=None,
        data_collator=SoftLabelDataCollator(tokenizer),
    )

    trainer.train()

    model = model.merge_and_unload()
    model.save_pretrained("./soft_distill_inkuba_output")
    tokenizer.save_pretrained("./soft_distill_inkuba_output")

    return model, tokenizer

# ---------------------------
# 8. Run training
# ---------------------------
if __name__ == "__main__":
    train_soft_distilled_lora_model(sentiment_xnli_raw, model_name="lelapa/InkubaLM-0.4B")




✅ Preprocessing data...
✅ Loading model and tokenizer...
trainable params: 524,288 || all params: 664,684,544 || trainable%: 0.0789
✅ Tokenizing dataset...


Map:   0%|          | 0/600 [00:00<?, ? examples/s]

✅ Filtering out incomplete samples...


Filter:   0%|          | 0/600 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


✅ Final dataset size: 600 examples
✅ Wrapping in PyTorch dataset...
🚀 Starting training...


Step,Training Loss
10,1.571500
20,1.574400
30,1.501500
40,1.464900
50,1.314000
60,1.241700
70,1.364400
80,1.449500
90,1.166400
100,1.216100


/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/bnb.py:355: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [86]:
import os
output_path = "/content/drive/MyDrive/InkubaLM-Challenge/Output/distillation"
os.makedirs(output_path, exist_ok=True)

print(os.path.exists(output_path))  # Should return True
print(output_path)

True
/content/drive/MyDrive/InkubaLM-Challenge/Output/distillation


In [65]:
model = AutoModelForCausalLM.from_pretrained("./soft_distill_inkuba_output").to("cuda")
tokenizer = AutoTokenizer.from_pretrained("lelapa/InkubaLM-0.4B")
tokenizer.pad_token = tokenizer.eos_token  # Ensure padding is defined


`low_cpu_mem_usage` was None, now default to True since model is quantized.


In [76]:
sentiment_xnli_raw[0]

{'ID': 'ID_f3c74c7b_sentiment_test__hausa',
 'task': 'sentiment',
 'instruction': "Your task is to do sentiment analysis. Your output must be one word: positive, negative, or neutral. Don't output any explanation.",
 'input': '@user ynxu fha da kanada kudi shikenan duk kayan nan zasu iya zama naka no🧢',
 'output': 'negative',
 'soft_label': {'positive': 0.0, 'neutral': 6e-05, 'negative': 0.99844},
 'lang': 'hausa',
 'prompt': "Your task is to do sentiment analysis. Your output must be one word: positive, negative, or neutral. Don't output any explanation.\n@user ynxu fha da kanada kudi shikenan duk kayan nan zasu iya zama naka no🧢"}

In [100]:
def main(
    model,
    tokenizer,
    BASE_PROMPT,  # e.g., "{}\n### Response:"
    task_instruction,
    dataset,
    csv_file_path,
    custom_instruct=False,
    sample_size=4,
    max_new_tokens=100,
    seed=42,
    do_sample=True,
    min_length=None,
    use_cache=True,
    top_p=1.0,
    temperature=0.5,
    top_k=5,
    repetition_penalty=1.2,
    length_penalty=1,
    debug_labels=False,
    **kwargs,
):
    actual_model = getattr(model, "model", model)
    actual_model.eval()

    with open(csv_file_path, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow([
            "ID", "Instruction", "Input Text", "Response", "Log-Likelihoods", "Targets", "Task", "Langs",
        ])

        for i, item in enumerate(dataset):
            if i >= sample_size:
                break

            instruction = item["instruction"] if not custom_instruct else task_instruction
            input_text = item["inputs"]
            target_label = str(item.get("targets", ""))
            langs = item.get("langs", "")
            task = item.get("task", "xnli")
            identity = item["ID"]

            # Align formatting
            prompt_text = f"{instruction}\n{input_text}".strip()
            full_prompt = BASE_PROMPT.format(prompt_text)

            if task == "mmt":
                batch = tokenizer(full_prompt, return_tensors="pt").to(model.device)
                with torch.no_grad():
                    outputs = model.generate(
                        **batch,
                        max_new_tokens=max_new_tokens,
                        do_sample=do_sample,
                        top_p=top_p,
                        temperature=temperature,
                        min_length=min_length,
                        use_cache=use_cache,
                        top_k=top_k,
                        repetition_penalty=repetition_penalty,
                        length_penalty=length_penalty,
                        **kwargs,
                    )

                decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)[len(full_prompt):].strip()
                response = decoded_output
                log_likelihoods = []

            else:
                if task == "xnli":
                    label_texts = ["Kweli", "Wala siyo", "Uongo"] if langs == "swa" else (
                        ["Gaskiya", "Tsaka-tsaki", "Karya"] if langs == "hau" else ["True", "Neither", "False"]
                    )
                else:  # sentiment
                    label_texts = ["Kyakkyawa", "Tsaka-tsaki", "Korau"] if langs == "hausa" else ["Chanya", "Wastani", "Hasi"]

                label_to_index = {label: idx for idx, label in enumerate(label_texts)}
                log_likelihoods = []

                for label in label_texts:
                    if debug_labels:
                        print(f"Label: {label} → Tokens: {tokenizer.tokenize(label)}")

                    prompt_tokens = tokenizer(full_prompt, return_tensors="pt").to(model.device)
                    label_tokens = tokenizer(label, return_tensors="pt").to(model.device)

                    input_ids = torch.cat([prompt_tokens["input_ids"], label_tokens["input_ids"][:, 1:]], dim=1)
                    attention_mask = torch.cat([prompt_tokens["attention_mask"], label_tokens["attention_mask"][:, 1:]], dim=1)

                    with torch.no_grad():
                        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                        log_probs = log_softmax(logits, dim=-1)

                    label_token_ids = label_tokens["input_ids"][0][1:]  # remove BOS
                    label_start = input_ids.size(1) - label_token_ids.size(0)
                    token_log_probs = log_probs[0, label_start:label_start + len(label_token_ids)]
                    selected_log_probs = token_log_probs.gather(1, label_token_ids.unsqueeze(1)).squeeze(1)

                    normalized_log_prob = selected_log_probs.sum().item() / len(label_token_ids)
                    log_likelihoods.append(normalized_log_prob)

                best_index = int(torch.argmax(torch.tensor(log_likelihoods)))
                response = label_to_index[label_texts[best_index]]

            writer.writerow([
                identity,
                instruction,
                input_text,
                response,
                log_likelihoods,
                target_label,
                task,
                langs,
            ])




In [77]:
BASE_PROMPT = "{}\n### Response:"

sentiment_test = pd.read_parquet("hf://datasets/lelapa/SentimentTest/data/train-00000-of-00001.parquet")
hau_sent_df = sentiment_test.loc[sentiment_test['langs'] == 'hausa']
swa_sent_df = sentiment_test.loc[sentiment_test['langs'] == 'swahili']
hau_sent_dset = Dataset.from_pandas(hau_sent_df)
swa_sent_dset = Dataset.from_pandas(swa_sent_df)

In [81]:
xnli_test = pd.read_parquet("hf://datasets/lelapa/XNLITest/data/train-00000-of-00001.parquet")
hau_xnli_df = xnli_test.loc[xnli_test['langs'] == 'hau']
swa_xnli_df = xnli_test.loc[xnli_test['langs'] == 'swa']
hau_xnli_dset = Dataset.from_pandas(hau_xnli_df)
swa_xnli_dset = Dataset.from_pandas(swa_xnli_df)

In [83]:
swa_xnli_dset

Dataset({
    features: ['ID', 'langs', 'premise', 'inputs', 'instruction', 'targets', '__index_level_0__'],
    num_rows: 150
})

In [114]:
instruction = "Your task is to do sentiment analysis. Your output must be one word: positive, negative, or neutral. Don't output any explanation."
main(
    model=model,
    tokenizer=tokenizer,
    BASE_PROMPT="{}\n### Response:",  # Matches how it was trained!
    task_instruction= instruction,
    dataset=swa_sent_dset,
    csv_file_path= os.path.join(output_path,"swa_sentiment_prediction_dev.csv"),
    custom_instruct=False,
    sample_size=100,
    max_new_tokens=5,
    temperature=0.5,
    debug_labels=False
)

In [113]:
instruction = "Your task is to do sentiment analysis. Your output must be one word: positive, negative, or neutral. Don't output any explanation."
main(
    model=model,
    tokenizer=tokenizer,
    BASE_PROMPT="{}\n### Response:",  # Matches how it was trained!
    task_instruction= instruction,
    dataset=hau_sent_dset,
    csv_file_path= os.path.join(output_path,"hau_sentiment_prediction_dev.csv"),
    custom_instruct=False,
    sample_size=100,
    max_new_tokens=5,
    temperature=0.5,
    debug_labels=False
)


In [109]:
task_instruction = (
    "Your task is to determine if the claim is entailed by the premise. "
    "Your output must be one word: true, false, or neither. Don't output any explanation."
)

main(
    model=model,
    tokenizer=tokenizer,
    BASE_PROMPT="{}\n### Response:",  # Matches how it was trained!
    task_instruction= task_instruction,
    dataset=swa_xnli_dset,
    csv_file_path= os.path.join(output_path, "swa_xnli_prediction_dev.csv"),
    custom_instruct=False,
    sample_size=100,
    max_new_tokens=5,
    temperature=0.5,
    debug_labels=False
)


In [105]:
task_instruction = (
    "Your task is to determine if the claim is entailed by the premise. "
    "Your output must be one word: true, false, or neither. Don't output any explanation."
)

main(
    model=model,
    tokenizer=tokenizer,
    BASE_PROMPT="{}\n### Response:",  # Matches how it was trained!
    task_instruction= task_instruction,
    dataset=hau_xnli_dset,
    csv_file_path= os.path.join(output_path, "hau_xnli_prediction_dev.csv"),
    custom_instruct=False,
    sample_size=100,
    max_new_tokens=5,
    temperature=0.5,
    debug_labels=False
)



In [110]:
swa_test = pd.read_csv(os.path.join(output_path,"swa_xnli_prediction_dev.csv"))
hau_test = pd.read_csv(os.path.join(output_path,"hau_xnli_prediction_dev.csv"))

In [120]:
swa_test['Response'].unique()

array([1, 2])

In [121]:
hau_test['Response'].unique()

array([2])

In [116]:
swa_sent = pd.read_csv(os.path.join(output_path,"swa_sentiment_prediction_dev.csv"))
hau_sent = pd.read_csv(os.path.join(output_path,"hau_sentiment_prediction_dev.csv"))

In [118]:
swa_sent['Response'].unique()

array([2, 1])

In [119]:
hau_sent['Response'].unique()

array([0, 1])

In [122]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def analyze_predictions(csv_file_path):
    df = pd.read_csv(csv_file_path)
    df = df[df['Task'] == 'xnli']

    y_true = df['Targets'].astype(int)
    y_pred = df['Response'].astype(int)

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred, labels=[0,1,2])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["0 (True)", "1 (Neither)", "2 (False)"])
    disp.plot(cmap='Blues')
    plt.title("Confusion Matrix (XNLI)")
    plt.show()

    # Margin Histogram
    def get_margin(row):
        scores = eval(row['Log-Likelihoods'])  # Convert string to list
        top2 = sorted(scores, reverse=True)[:2]
        return top2[0] - top2[1]

    df["margin"] = df.apply(get_margin, axis=1)
    plt.hist(df["margin"], bins=30, edgecolor="black")
    plt.title("Prediction Margin (Top1 - Top2)")
    plt.xlabel("Log-likelihood margin")
    plt.ylabel("Count")
    plt.show()

In [123]:
analyze_predictions(os.path.join(output_path, "swa_xnli_prediction_dev.csv"))

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer